# autofe_bench no Colab
Compara **sem_fe, CAAFE, LLM-FE, GuidedFE** (e o seu método) sob o mesmo protocolo.
Rode as células **em ordem, uma de cada vez** (evite 'Executar tudo'): a 1ª pede autorização do Drive.

In [ ]:
# 1) Ambiente. Use UMA das opções de instalação.
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scikit-learn requests scipy ucimlrepo

REPO_URL = ''   # ex.: 'https://github.com/SEU_USUARIO/autofe-bench.git'  (deixe vazio para usar o zip do Drive)
if REPO_URL:
    !rm -rf /content/autofe-bench && git clone -q {REPO_URL} /content/autofe-bench
    %cd /content/autofe-bench
    !pip install -q -e .
else:
    !unzip -oq /content/drive/MyDrive/autofe/autofe-bench.zip -d /content   # ajuste o caminho do zip
    %cd /content/autofe-bench
    !pip install -q -e .


## 2) Parâmetros do experimento (edite aqui)

In [ ]:
PROVEDOR      = 'ollama'                 # 'ollama' (GPU do Colab, sem cota) ou 'groq' (precisa de GROQ_API_KEY nos Secrets)
OLLAMA_MODEL  = 'qwen3-coder:30b'        # veja `ollama list` / ollama.com/library
GROQ_MODEL    = 'openai/gpt-oss-120b'    # veja list_groq_models()
DATASETS      = ['titanic', 'adult', 'bank_marketing', 'cdc_diabetes']   # tambem: 'openml:ID' ou 'csv:CAMINHO:ALVO:TIPO'
AUTOFES       = ['caafe', 'llmfe', 'guided']   # o baseline 'sem_fe' e incluido automaticamente (pode listar, nao duplica)
PROTOCOLO     = 'dev'                    # 'smoke' (teste), 'dev' (desenvolvimento) ou 'v1' (oficial; nao misture com dev)
STORE  = '/content/results.jsonl'                        # local (rapido)
MIRROR = '/content/drive/MyDrive/autofe/results.jsonl'   # copia no Drive: sobrevive a quedas


## 3) Só para Ollama: instalar e subir o servidor
Use *Ambiente de execução → Alterar tipo → GPU*. O download do modelo se repete a cada sessão nova.

In [ ]:
if PROVEDOR == 'ollama':
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    import subprocess, time
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    time.sleep(8)
    !ollama pull {OLLAMA_MODEL}
    !ollama list


## 4) Cliente do LLM e teste de contrato (poucas chamadas)

In [ ]:
import os
from autofe_bench.llm import GroqClient, OllamaClient, list_groq_models
if PROVEDOR == 'groq':
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print(list_groq_models())   # o modelo escolhido precisa estar na lista
    client = GroqClient(GROQ_MODEL, rpm=25, tpm=6000)   # confira seus limites em console.groq.com/settings/limits
else:
    client = OllamaClient(OLLAMA_MODEL, num_ctx=8192)

from autofe_bench.check import check_method
from autofe_bench.methods import resolve_method
for m in AUTOFES:
    if resolve_method(m).needs_llm:
        print('==', m); check_method(m, 'breast_cancer', client=client)


## 5) Estimativa de tempo/tokens

In [ ]:
from autofe_bench.runner import estimate
print(estimate(AUTOFES, DATASETS, PROTOCOLO, STORE, tpd=None)['text'])   # com Groq: tpd=SEU_LIMITE_DIARIO


## 6) Rodar (retomável)
Se cair ou a cota acabar, rode esta célula de novo: ela continua de onde parou.

In [ ]:
from autofe_bench.runner import run_benchmark
run_benchmark(AUTOFES, DATASETS, protocol=PROTOCOLO, client=client, store_path=STORE, mirror_path=MIRROR, log_dir='/content/logs')


## 7) Relatório
Use `model='lr'`, `'rf'` ou `'hgb'` para ver cada modelo final.

In [ ]:
from autofe_bench.report import render
for mdl in ['lr', 'rf', 'hgb']:
    print(render(STORE, model=mdl)); print('\n' + '=' * 70 + '\n')


## 8) Baixar o resultado
Envie o `results.jsonl` para quem vai juntar os resultados do grupo (`python -m autofe_bench merge a.jsonl b.jsonl -o todos.jsonl`).

In [ ]:
from google.colab import files
files.download(STORE)